# Notebook 2 v3 - Data Conditioning, Full-Dataset-Aware Sampling, and Corrected Repeat Proxy

This notebook prepares the analysis-ready data used by modeling and clustering.

1. Replaces first-300k-row sampling with representative row-group sampling across the full parquet.
2. Builds a repeat complaint proxy from complaint type, geography, and month instead of complaint type alone.
3. Keeps full-dataset-aware aggregate outputs while saving a memory-safe modeling sample.
4. Saves clear transformation rules and handoff files for Notebooks 3 and 4.


In [ ]:
import os
import gc
import json
import math
import time
import shutil
import zipfile
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

try:
    import pyarrow.parquet as pq
except Exception:
    !pip -q install pyarrow
    import pyarrow.parquet as pq


# Upload run_mainfest.json from noteook 1


In [ ]:
# USER CONFIGURATION

# If Notebook 1 v3 was run, this file should exist.
MANIFEST_PATH = "/content/run_manifest.json"

# Fallback source path if the manifest is not available.
# SOURCE_PARQUET_FALLBACK = "/content/drive/MyDrive/project_data/Capstone Course Project Files/Data/nyc_311_FINAL_UPDATED.parquet"
SOURCE_PARQUET_FALLBACK = "/content/drive/MyDrive/project_data/Capstone Course Project Files/Data/enriched_cleaned_nyc_311.parquet"

OUTPUT_DIR = Path("/content/notebook2_v3_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Analysis sample size for Notebook 3 and 4.
# 400k to 600k is usually a good Standard Colab range.
ANALYSIS_SAMPLE_TARGET_ROWS = 500_000

# Batch/row group sampling settings.
RANDOM_STATE = 42

# Repeat proxy threshold.
# A record is flagged as repeat-like when at least this many similar records exist
# in the same complaint/geography/month group in the full dataset aggregate.
REPEAT_GROUP_THRESHOLD = 3

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
print("Output directory:", OUTPUT_DIR)


Output directory: /content/notebook2_v3_outputs


In [ ]:
# LOAD MANIFEST AND SOURCE PARQUET

manifest = {}
if os.path.exists(MANIFEST_PATH):
    with open(MANIFEST_PATH, "r") as f:
        manifest = json.load(f)

SOURCE_PARQUET = manifest.get("source_parquet", SOURCE_PARQUET_FALLBACK)

if SOURCE_PARQUET.startswith("/content/drive"):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print("Drive mount skipped or not running in Colab:", e)

if not os.path.exists(SOURCE_PARQUET):
    raise FileNotFoundError(f"Source parquet not found: {SOURCE_PARQUET}")

pf = pq.ParquetFile(SOURCE_PARQUET)
print("Rows:", f"{pf.metadata.num_rows:,}")
print("Row groups:", pf.metadata.num_row_groups)
print("Columns:", len(pf.schema_arrow.names))


Mounted at /content/drive
Rows: 20,448,149
Row groups: 20
Columns: 48


In [ ]:
# COLUMN NORMALIZATION AND SELECTION HELPERS

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        pd.Index(df.columns)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace(r"[^0-9a-zA-Z_]+", "", regex=True)
    )
    return df

source_columns_raw = list(pf.schema_arrow.names)
source_columns_norm = list(normalize_columns(pd.DataFrame(columns=source_columns_raw)).columns)
raw_by_norm = dict(zip(source_columns_norm, source_columns_raw))

def raw_cols_for(norm_cols):
    """Map normalized column names back to raw parquet names when present."""
    return [raw_by_norm[c] for c in norm_cols if c in raw_by_norm]

# Core columns needed for analysis and modeling.
# Missing columns are skipped safely.
CORE_COLUMNS_NORM = [
    "unique_key", "created_date", "closed_date", "status", "agency", "agency_name",
    "complaint_type", "descriptor", "location_type", "incident_zip", "incident_address",
    "street_name", "cross_street_1", "cross_street_2", "address_type", "city",
    "borough", "latitude", "longitude", "open_data_channel_type",
    "resolution_description", "community_board", "bbl", "park_facility_name", "bridge_highway_name",
    "zip_code", "resolution_time_hours","year", "month", "day", "hour", "yearmon",
    "season", "time_of_day", "is_weekend", "is_holiday",
    "possible_repeat_count",  "permit_count", "white_pct", "black_pct", "hispanic_pct", "not_citizen_pct",
    "below_poverty_pct", "household_inc", "PRCP", "SNOW", "TMAX", "TMIN","precinct", "pol_lat", "pol_lon",
    "distance_to_precinct"
]

CORE_COLUMNS_RAW = raw_cols_for(CORE_COLUMNS_NORM)
print("Columns to read:", CORE_COLUMNS_RAW)


Columns to read: ['unique_key', 'created_date', 'closed_date', 'agency_name', 'complaint_type', 'descriptor', 'location_type', 'address_type', 'borough', 'latitude', 'longitude', 'open_data_channel_type', 'community_board', 'park_facility_name', 'bridge_highway_name', 'zip_code', 'resolution_time_hours', 'year', 'month', 'day', 'hour', 'yearmon', 'season', 'time_of_day', 'is_weekend', 'is_holiday', 'possible_repeat_count', 'permit_count', 'white_pct', 'black_pct', 'hispanic_pct', 'not_citizen_pct', 'below_poverty_pct', 'household_inc', 'precinct', 'pol_lat', 'pol_lon', 'distance_to_precinct']


In [ ]:
# CHUNK CONDITIONING FUNCTION

def condition_chunk(df: pd.DataFrame) -> pd.DataFrame:
    """Clean and engineer features for one chunk.

    This function is used consistently for full-dataset aggregates and sampled data.
    """
    df = normalize_columns(df)

    # Standardize key categorical text columns.
    cat_cols = [
        "status", "agency", "agency_name", "complaint_type", "descriptor",
        "location_type", "incident_zip", "address_type", "city", "borough",
        "open_data_channel_type", "community_board"
    ]
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].astype("string").str.strip().str.upper().fillna("UNKNOWN")
            df.loc[df[c].isin(["", "NAN", "NONE", "NULL"]), c] = "UNKNOWN"

    # Datetime parsing.
    if "created_date" in df.columns:
        df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")
    if "closed_date" in df.columns:
        df["closed_date"] = pd.to_datetime(df["closed_date"], errors="coerce")

    # Numeric coordinates.
    for c in ["latitude", "longitude"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Response time and closed flags.
    if {"created_date", "closed_date"}.issubset(df.columns):
        df["response_time_hours"] = (df["closed_date"] - df["created_date"]).dt.total_seconds() / 3600
        df.loc[df["response_time_hours"] < 0, "response_time_hours"] = np.nan
        df["closed_same_day_flag"] = (
            df["created_date"].dt.date == df["closed_date"].dt.date
        ).astype("Int64")
    else:
        df["response_time_hours"] = np.nan
        df["closed_same_day_flag"] = pd.Series([pd.NA] * len(df), dtype="Int64")

    # Time features.
    if "created_date" in df.columns:
        df["created_year"] = df["created_date"].dt.year.astype("Int64")
        df["created_month"] = df["created_date"].dt.month.astype("Int64")
        df["created_ym"] = df["created_date"].dt.to_period("M").astype(str)
        df["created_dayofweek"] = df["created_date"].dt.dayofweek.astype("Int64")
        df["created_hour"] = df["created_date"].dt.hour.astype("Int64")
        df["is_weekend"] = df["created_dayofweek"].isin([5, 6]).astype(int)
        month = df["created_month"]
        df["season"] = np.select(
            [month.isin([12, 1, 2]), month.isin([3, 4, 5]), month.isin([6, 7, 8]), month.isin([9, 10, 11])],
            ["WINTER", "SPRING", "SUMMER", "FALL"],
            default="UNKNOWN"
        )
    else:
        df["created_ym"] = "UNKNOWN"
        df["is_weekend"] = 0
        df["season"] = "UNKNOWN"

    # Geography fields for repeat proxy.
    if "incident_zip" not in df.columns:
        df["incident_zip"] = "UNKNOWN"
    if "borough" not in df.columns:
        df["borough"] = "UNKNOWN"

    # Rounded coordinate cells. These are broad enough for repeat-like clustering but not too specific.
    if {"latitude", "longitude"}.issubset(df.columns):
        df["lat_round3"] = df["latitude"].round(3)
        df["lon_round3"] = df["longitude"].round(3)
        df["geo_cell"] = df["lat_round3"].astype("string") + "," + df["lon_round3"].astype("string")
        df.loc[df["latitude"].isna() | df["longitude"].isna(), "geo_cell"] = "ZIP_" + df["incident_zip"].astype("string")
    else:
        df["geo_cell"] = "ZIP_" + df["incident_zip"].astype("string")

    # Correct repeat key: complaint type plus geography plus month.
    # This replaces the old invalid logic that treated any repeated complaint type as repeat activity.
    if "complaint_type" not in df.columns:
        df["complaint_type"] = "UNKNOWN"
    df["repeat_proxy_key"] = (
        df["complaint_type"].astype("string") + "|" +
        df["borough"].astype("string") + "|" +
        df["geo_cell"].astype("string") + "|" +
        df["created_ym"].astype("string")
    )

    if "incident_zip" not in df.columns and "zip_code" in df.columns:
      df["incident_zip"] = df["zip_code"]

    return df


In [ ]:
# REPRESENTATIVE ROW-GROUP SAMPLE ACROSS FULL PARQUET
# This is the core fix. We sample from every row group, so the modeling sample is not limited
# to the first rows of a date-sorted parquet file.

rng = np.random.default_rng(RANDOM_STATE)
n_groups = pf.metadata.num_row_groups
rows_per_group = max(1, math.ceil(ANALYSIS_SAMPLE_TARGET_ROWS / n_groups))

sample_pieces = []
for rg in range(n_groups):
    df = pf.read_row_group(rg, columns=CORE_COLUMNS_RAW).to_pandas()
    df = condition_chunk(df)

    if len(df) > rows_per_group:
        df = df.sample(n=rows_per_group, random_state=int(rng.integers(0, 1_000_000)))
    sample_pieces.append(df)

    if rg % 10 == 0:
        print(f"Sampled row group {rg+1}/{n_groups}")
    del df
    gc.collect()

analysis_sample = pd.concat(sample_pieces, ignore_index=True)
if len(analysis_sample) > ANALYSIS_SAMPLE_TARGET_ROWS:
    analysis_sample = analysis_sample.sample(n=ANALYSIS_SAMPLE_TARGET_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)

print("Representative sample rows:", f"{len(analysis_sample):,}")
print("Date range:", analysis_sample["created_date"].min(), "to", analysis_sample["created_date"].max())
display(analysis_sample.head())


Sampled row group 1/20
Sampled row group 11/20
Representative sample rows: 500,000
Date range: 2020-01-01 00:04:47 to 2026-04-28 01:36:23


,unique_key,created_date,closed_date,agency_name,complaint_type,descriptor,location_type,address_type,borough,latitude,longitude,open_data_channel_type,community_board,park_facility_name,bridge_highway_name,zip_code,resolution_time_hours,year,month,day,hour,yearmon,season,time_of_day,is_weekend,is_holiday,possible_repeat_count,permit_count,white_pct,black_pct,hispanic_pct,not_citizen_pct,below_poverty_pct,household_inc,precinct,pol_lat,pol_lon,distance_to_precinct,response_time_hours,closed_same_day_flag,created_year,created_month,created_ym,created_dayofweek,created_hour,incident_zip,lat_round3,lon_round3,geo_cell,repeat_proxy_key
0,68366989,2026-03-18 18:56:08,NaT,DEPARTMENT OF HOUSING PRESERVATION AND DEVELOP...,ELECTRIC,LIGHTING,RESIDENTIAL BUILDING,ADDRESS,BRONX,40.883980,-73.866196,PHONE,12 BRONX,NaN,NaN,10467,NaN,2026,3,18,18.0,2026-03,SPRING,evening,0,False,2.0,78.0,NaN,NaN,NaN,NaN,NaN,NaN,Precinct 47,40.887511,-73.847554,1.006188,NaN,0,2026,3,2026-03,2,18,UNKNOWN,40.884,-73.866,"40.884,-73.866","ELECTRIC|BRONX|40.884,-73.866|2026-03"
1,66884919,2025-11-19 15:40:41,2025-11-20 14:39:29,DEPARTMENT OF HOMELESS SERVICES,ENCAMPMENT,UNKNOWN,STREET/SIDEWALK,ADDRESS,MANHATTAN,40.731456,-73.985578,MOBILE,03 MANHATTAN,NaN,NaN,10003,22.980000,2025,11,19,15.0,2025-11,FALL,afternoon,0,False,NaN,185.0,NaN,NaN,NaN,NaN,NaN,NaN,Precinct 9,40.726538,-73.987829,0.359339,22.980000,0,2025,11,2025-11,2,15,UNKNOWN,40.731,-73.986,"40.731,-73.986","ENCAMPMENT|MANHATTAN|40.731,-73.986|2025-11"
2,66835073,2025-11-14 18:47:00,2025-11-19 00:42:00,DEPARTMENT OF ENVIRONMENTAL PROTECTION,NOISE,NOISE: ALARMS (NR3),UNKNOWN,ADDRESS,QUEENS,40.750166,-73.940947,ONLINE,02 QUEENS,NaN,NaN,11101,101.916667,2025,11,14,18.0,2025-11,FALL,evening,0,False,3.0,126.0,NaN,NaN,NaN,NaN,NaN,NaN,Precinct 108,40.743040,-73.954744,0.875255,101.916667,0,2025,11,2025-11,4,18,UNKNOWN,40.750,-73.941,"40.75,-73.941","NOISE|QUEENS|40.75,-73.941|2025-11"
3,66900233,2025-11-19 12:50:00,2025-11-20 04:14:00,DEPARTMENT OF SANITATION,DERELICT VEHICLES,DERELICT VEHICLES,STREET,ADDRESS,QUEENS,40.721762,-73.908806,PHONE,05 QUEENS,NaN,NaN,11378,15.400000,2025,11,19,12.0,2025-11,FALL,afternoon,0,False,3.0,35.0,NaN,NaN,NaN,NaN,NaN,NaN,Precinct 104,40.704504,-73.893497,1.436725,15.400000,0,2025,11,2025-11,2,12,UNKNOWN,40.722,-73.909,"40.722,-73.909","DERELICT VEHICLES|QUEENS|40.722,-73.909|2025-11"
4,67135176,2025-12-10 14:22:15,NaT,DEPARTMENT OF SANITATION,GRAFFITI,GRAFFITI,MIXED USE,ADDRESS,BROOKLYN,40.597549,-73.985409,UNKNOWN,11 BROOKLYN,NaN,NaN,11223,NaN,2025,12,10,14.0,2025-12,WINTER,afternoon,0,False,2.0,119.0,NaN,NaN,NaN,NaN,NaN,NaN,Precinct 62,40.602577,-74.003000,0.988082,NaN,0,2025,12,2025-12,2,14,UNKNOWN,40.598,-73.985,"40.598,-73.985","GRAFFITI|BROOKLYN|40.598,-73.985|2025-12"


In [ ]:
# FULL-DATASET REPEAT PROXY AGGREGATE
# This scans the full parquet in row groups and counts repeat_proxy_key frequency.
# It stores only grouped counts, not all 20M records.

repeat_chunks_dir = OUTPUT_DIR / "repeat_key_chunks"
repeat_chunks_dir.mkdir(exist_ok=True)

repeat_chunk_files = []
for rg in range(n_groups):
    df = pf.read_row_group(rg, columns=CORE_COLUMNS_RAW).to_pandas()
    df = condition_chunk(df)
    counts = df.groupby("repeat_proxy_key", dropna=False).size().reset_index(name="repeat_group_count")
    chunk_path = repeat_chunks_dir / f"repeat_counts_rg_{rg:04d}.csv"
    counts.to_csv(chunk_path, index=False)
    repeat_chunk_files.append(chunk_path)

    if rg % 10 == 0:
        print(f"Built repeat counts for row group {rg+1}/{n_groups}")
    del df, counts
    gc.collect()

repeat_counts = pd.concat([pd.read_csv(p) for p in repeat_chunk_files], ignore_index=True)
repeat_counts = repeat_counts.groupby("repeat_proxy_key", as_index=False)["repeat_group_count"].sum()
repeat_counts.to_csv(OUTPUT_DIR / "full_dataset_repeat_proxy_counts.csv", index=False)
print("Repeat key groups:", f"{len(repeat_counts):,}")
display(repeat_counts.sort_values("repeat_group_count", ascending=False).head(20))


Built repeat counts for row group 1/20
Built repeat counts for row group 11/20
Repeat key groups: 9,721,121


,repeat_proxy_key,repeat_group_count
5299662,"NOISE - RESIDENTIAL|BRONX|40.892,-73.86|2025-01",45412
5299673,"NOISE - RESIDENTIAL|BRONX|40.892,-73.86|2025-12",29932
5299501,"NOISE - RESIDENTIAL|BRONX|40.892,-73.859|2022-07",24030
5299661,"NOISE - RESIDENTIAL|BRONX|40.892,-73.86|2024-12",20615
5299503,"NOISE - RESIDENTIAL|BRONX|40.892,-73.859|2022-09",18690
5299671,"NOISE - RESIDENTIAL|BRONX|40.892,-73.86|2025-10",18678
5299658,"NOISE - RESIDENTIAL|BRONX|40.892,-73.86|2024-09",15907
5299499,"NOISE - RESIDENTIAL|BRONX|40.892,-73.859|2022-05",15422
5299672,"NOISE - RESIDENTIAL|BRONX|40.892,-73.86|2025-11",14727
5299490,"NOISE - RESIDENTIAL|BRONX|40.892,-73.859|2021-07",13808


In [ ]:
# APPLY CORRECTED REPEAT PROXY TO ANALYSIS SAMPLE

analysis_sample = analysis_sample.merge(repeat_counts, on="repeat_proxy_key", how="left")
analysis_sample["repeat_group_count"] = analysis_sample["repeat_group_count"].fillna(1).astype(int)
analysis_sample["repeat_complaint_proxy"] = (analysis_sample["repeat_group_count"] >= REPEAT_GROUP_THRESHOLD).astype(int)

repeat_summary = analysis_sample["repeat_complaint_proxy"].value_counts(dropna=False).reset_index()
repeat_summary.columns = ["repeat_complaint_proxy", "sample_count"]
repeat_summary.to_csv(OUTPUT_DIR / "repeat_proxy_sample_summary.csv", index=False)
display(repeat_summary)


,repeat_complaint_proxy,sample_count
0,1,261781
1,0,238219


In [ ]:
# FINAL MODELING FILTERS AND TARGETS
# Keep closed records with valid non-negative response time for modeling.

model_sample = analysis_sample.copy()
model_sample = model_sample[model_sample["response_time_hours"].notna()].copy()
model_sample = model_sample[model_sample["response_time_hours"] >= 0].copy()

# Winsorized response time is useful for regression because the raw distribution is very skewed.
upper_cap = model_sample["response_time_hours"].quantile(0.99)
model_sample["response_time_hours_capped_p99"] = model_sample["response_time_hours"].clip(upper=upper_cap)

# True delay-risk target for Notebook 3.
# This aligns better with the final report language than same-day closure alone.
DELAY_HOURS_THRESHOLD = 48
model_sample["delay_risk_48h_flag"] = (model_sample["response_time_hours"] > DELAY_HOURS_THRESHOLD).astype(int)
model_sample["delay_risk_p75_flag"] = (model_sample["response_time_hours"] > model_sample["response_time_hours"].quantile(0.75)).astype(int)

print("Model sample rows:", f"{len(model_sample):,}")
print("Response time p99 cap:", upper_cap)
print("Delay >48h rate:", model_sample["delay_risk_48h_flag"].mean())
print("Date range:", model_sample["created_date"].min(), "to", model_sample["created_date"].max())


Model sample rows: 487,454
Response time p99 cap: 12369.487647222197
Delay >48h rate: 0.3285622848514937
Date range: 2020-01-01 00:04:47 to 2026-04-28 01:12:01


In [ ]:
# SAVE ANALYSIS-READY OUTPUTS

analysis_path = OUTPUT_DIR / "analysis_ready_sample_v3.parquet"
model_path = OUTPUT_DIR / "model_ready_sample_v3.parquet"

analysis_sample.to_parquet(analysis_path, index=False)
model_sample.to_parquet(model_path, index=False)

field_list = pd.DataFrame({"column_name": model_sample.columns})
field_list.to_csv(OUTPUT_DIR / "final_model_field_list_v3.csv", index=False)

transformation_rules = pd.DataFrame([
    {"step": "representative_sampling", "description": "Sampled across all parquet row groups instead of first-N rows."},
    {"step": "response_time_hours", "description": "closed_date minus created_date in hours; negative values set to missing."},
    {"step": "closed_same_day_flag", "description": "1 when created_date and closed_date are the same calendar day."},
    {"step": "delay_risk_48h_flag", "description": "1 when response_time_hours exceeds 48 hours."},
    {"step": "repeat_complaint_proxy", "description": f"1 when complaint_type + borough + geography cell + month appears at least {REPEAT_GROUP_THRESHOLD} times in full-dataset aggregate."},
    {"step": "response_time_hours_capped_p99", "description": "Response time capped at the sample 99th percentile for regression stability."},
])
transformation_rules.to_csv(OUTPUT_DIR / "transformation_rules_v3.csv", index=False)

handoff_manifest = {
    "run_timestamp": RUN_TS,
    "source_parquet": SOURCE_PARQUET,
    "analysis_ready_sample_path": str(analysis_path),
    "model_ready_sample_path": str(model_path),
    "repeat_counts_path": str(OUTPUT_DIR / "full_dataset_repeat_proxy_counts.csv"),
    "sample_rows": int(len(analysis_sample)),
    "model_rows": int(len(model_sample)),
    "sample_min_created_date": str(model_sample["created_date"].min()),
    "sample_max_created_date": str(model_sample["created_date"].max()),
    "delay_hours_threshold": DELAY_HOURS_THRESHOLD,
    "repeat_group_threshold": REPEAT_GROUP_THRESHOLD,
    "important_note": "Notebook 2 v3 creates representative samples across row groups and uses corrected repeat proxy logic."
}

with open(OUTPUT_DIR / "notebook2_v3_manifest.json", "w") as f:
    json.dump(handoff_manifest, f, indent=2)

# Copy convenient handoff files to /content.
shutil.copy(OUTPUT_DIR / "notebook2_v3_manifest.json", "/content/notebook2_v3_manifest.json")
print(json.dumps(handoff_manifest, indent=2))


{
  "run_timestamp": "20260512_100933",
  "source_parquet": "/content/drive/MyDrive/project_data/Capstone Course Project Files/Data/enriched_cleaned_nyc_311.parquet",
  "analysis_ready_sample_path": "/content/notebook2_v3_outputs/analysis_ready_sample_v3.parquet",
  "model_ready_sample_path": "/content/notebook2_v3_outputs/model_ready_sample_v3.parquet",
  "repeat_counts_path": "/content/notebook2_v3_outputs/full_dataset_repeat_proxy_counts.csv",
  "sample_rows": 500000,
  "model_rows": 487454,
  "sample_min_created_date": "2020-01-01 00:04:47",
  "sample_max_created_date": "2026-04-28 01:12:01",
  "delay_hours_threshold": 48,
  "repeat_group_threshold": 3,
  "important_note": "Notebook 2 v3 creates representative samples across row groups and uses corrected repeat proxy logic."
}


In [ ]:
# CREATE NOTEBOOK 2 OUTPUT ZIP

import shutil
from pathlib import Path

NOTEBOOK2_OUTPUT_DIR = Path("/content/notebook2_v3_outputs")
ZIP_BASE = Path("/content/notebook2_v3_outputs")

# This creates /content/notebook2_v3_outputs.zip
zip_path = shutil.make_archive(
    base_name=str(ZIP_BASE),
    format="zip",
    root_dir=str(NOTEBOOK2_OUTPUT_DIR)
)

print("Created zip:", zip_path)
print("Upload or download this file for Notebook 3:")
print(zip_path)

Created zip: /content/notebook2_v3_outputs.zip
Upload or download this file for Notebook 3:
/content/notebook2_v3_outputs.zip
